# AHM Movies — Free Processing Session
Run this notebook in Google Colab. It temporarily hosts the AHM Movies worker while this Colab session stays connected. No Render payment is needed.

**Before running:** choose Runtime → Change runtime type → T4 GPU (if available). Free GPU access is not guaranteed. Keep this notebook tab/session running while a movie is processing.

Your API keys are entered privately into notebook prompts and are not saved in this notebook. Never paste them into ChatGPT or GitHub.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg fonts-dejavu-core
!pip -q install fastapi==0.115.14 uvicorn==0.34.3 boto3==1.38.46 httpx==0.28.1 faster-whisper==1.1.1 edge-tts==7.2.0 python-multipart==0.0.20
!git clone --depth 1 --branch feature/movie-recap-rebuild https://github.com/socialboosthub/ahm-movies.git /content/ahm-movies
%cd /content/ahm-movies/worker
print('AHM Movies worker dependencies installed.')

## 2. Add your private service credentials
Enter your Cloudflare R2 values and Groq key when prompted. The worker token will be generated for you; copy it somewhere safe because the same value must be added to Vercel as `RECAP_WORKER_TOKEN`.

In [ ]:
import os, getpass, secrets, subprocess, pathlib, time, re, requests
os.environ['R2_ACCOUNT_ID'] = getpass.getpass('Cloudflare R2 Account ID: ')
os.environ['R2_ACCESS_KEY_ID'] = getpass.getpass('R2 Access Key ID: ')
os.environ['R2_SECRET_ACCESS_KEY'] = getpass.getpass('R2 Secret Access Key: ')
os.environ['R2_BUCKET'] = input('R2 bucket name: ').strip()
os.environ['GROQ_API_KEY'] = getpass.getpass('Groq API key: ')
os.environ['RECAP_WORKER_TOKEN'] = secrets.token_urlsafe(36)
print('\nYour worker token (copy this; you will add it to Vercel):\n')
print(os.environ['RECAP_WORKER_TOKEN'])
import torch
if torch.cuda.is_available():
    os.environ['WHISPER_DEVICE']='cuda'
    os.environ['WHISPER_COMPUTE_TYPE']='float16'
    print('\nGPU detected:', torch.cuda.get_device_name(0))
else:
    os.environ['WHISPER_DEVICE']='cpu'
    os.environ['WHISPER_COMPUTE_TYPE']='int8'
    print('\nNo GPU detected. CPU mode selected; full-movie processing may be very slow.')
os.environ['WHISPER_MODEL']='small'
print('Credentials loaded into this Colab session only.')

## 3. Start the worker and a temporary secure tunnel
The public URL below changes when you restart the notebook. You will need to update `RECAP_WORKER_URL` in the Vercel Preview environment each time the URL changes.

In [ ]:
import platform, urllib.request
arch = platform.machine().lower()
binary_url = 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
urllib.request.urlretrieve(binary_url, '/usr/local/bin/cloudflared')
os.chmod('/usr/local/bin/cloudflared', 0o755)
worker_log = open('/content/ahm-worker.log','w')
worker_proc = subprocess.Popen(['python','-m','uvicorn','movie_api:app','--host','127.0.0.1','--port','8000'], stdout=worker_log, stderr=subprocess.STDOUT, env=os.environ.copy())
for _ in range(60):
    try:
        if requests.get('http://127.0.0.1:8000/health', timeout=2).ok: break
    except Exception: pass
    time.sleep(2)
else: raise RuntimeError('Worker did not start. Check /content/ahm-worker.log')
tunnel_log = open('/content/ahm-tunnel.log','w')
tunnel_proc = subprocess.Popen(['/usr/local/bin/cloudflared','tunnel','--url','http://127.0.0.1:8000','--no-autoupdate'], stdout=tunnel_log, stderr=subprocess.STDOUT)
worker_url = None
for _ in range(60):
    time.sleep(2)
    tunnel_log.flush()
    log_text = pathlib.Path('/content/ahm-tunnel.log').read_text(errors='ignore')
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', log_text)
    if match:
        worker_url = match.group(0)
        break
if not worker_url: raise RuntimeError('Tunnel URL not found. Re-run this cell and inspect /content/ahm-tunnel.log')
print('AHM Movies worker URL:', worker_url)
print('Health check:', requests.get(worker_url + '/health', timeout=30).json())
print('\nKeep this Colab session running while processing movies.')

## 4. Connect Vercel Preview
In Vercel → `ahm-movies` → Settings → Environment Variables, add these for **Preview** (branch `feature/movie-recap-rebuild`):

- `RECAP_WORKER_URL` = the `https://....trycloudflare.com` URL printed above (no trailing slash)
- `RECAP_WORKER_TOKEN` = the generated worker token printed in section 2

Keep the existing R2 environment variables in Vercel. Redeploy the latest Preview after saving. Do not add these to Production yet. The tunnel URL is temporary and changes after a Colab restart.

**Important limitations:** Colab free sessions can disconnect or reclaim resources; a restart interrupts a running job. This is a free development/testing bridge, not always-on production hosting. Test first with a short movie clip before attempting a full-length film.

In [ ]:
print('Worker running:', worker_proc.poll() is None)
print('Tunnel running:', tunnel_proc.poll() is None)
print('Health:', requests.get(worker_url + '/health', timeout=20).json())